In [1]:
import requests

def download_book(url, skip=0):
    text = requests.get(url).text
    return text[skip:]

# Marcus Aurelius - Meditations
book1 = download_book("https://www.gutenberg.org/files/55317/55317-0.txt", skip=2000)

# Epictetus - Enchiridion
book2 = download_book("https://www.gutenberg.org/files/45898/45898-0.txt", skip=1500)

# Sun Tzu - Art of War
book3 = download_book("https://www.gutenberg.org/files/132/132-0.txt", skip=1500)

# Combine all three
text = book1 + book2 + book3

print(f"Total characters: {len(text)}")
print("\nSample:")
print(text[:300])

Total characters: 1034103

Sample:
lling of
Demons and the like; not to keep quails (for fighting or divination),
nor to run after such things; to suffer freedom of speech in others,
and to apply myself heartily to philosophy. Him also I must thank for
my hearing first Bacchius, then Tandasis and Marcianus; that I wrote
dialogue


In [2]:
# Find all unique characters in the text
chars = sorted(set(text))
vocab_size = len(chars)

print(f"Unique characters: {vocab_size}")
print(f"Characters: {''.join(chars[:50])}")

# Create two dictionaries
# char_to_idx: given a character, give me its number
# idx_to_char: given a number, give me its character
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}

# Example
print(f"\n'A' = {char_to_idx['A']}")
print(f"'z' = {char_to_idx['z']}")
print(f"{char_to_idx['A']} = '{idx_to_char[char_to_idx['A']]}'")

Unique characters: 109
Characters: 	
 !"$%&'()*,-./0123456789:;?@ABCDEFGHIJKLMNOPQRS

'A' = 31
'z' = 85
31 = 'A'


In [3]:
import torch
import torch.nn as nn

# Convert entire text into numbers
encoded = [char_to_idx[ch] for ch in text]
encoded = torch.tensor(encoded, dtype=torch.long)

print(f"Encoded length: {len(encoded)}")
print(f"First 10 numbers: {encoded[:10]}")
print(f"Which is: {''.join([idx_to_char[i.item()] for i in encoded[:10]])}")

# Create input and target sequences
seq_length = 100  # model reads 100 characters at a time

def get_batches(encoded, batch_size, seq_length):
    batches = []
    for i in range(0, len(encoded) - seq_length - 1, seq_length):
        x = encoded[i:i+seq_length]
        y = encoded[i+1:i+seq_length+1]
        batches.append((x, y))
    return batches

batches = get_batches(encoded, 32, seq_length)
print(f"\nTotal training sequences: {len(batches)}")

Encoded length: 1034103
First 10 numbers: tensor([71, 71, 68, 73, 66,  3, 74, 65,  2,  1])
Which is: lling of


Total training sequences: 10341


In [4]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()

        # Layer 1: Embedding - converts numbers to vectors
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Layer 2: LSTM - the memory layer
        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=0.3
        )

        # Layer 3: Output - converts LSTM output to character prediction
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        # x → embedding → lstm → fc → prediction
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size, device):
        # Initialize empty memory at start
        return (torch.zeros(2, batch_size, 256).to(device),
                torch.zeros(2, batch_size, 256).to(device))

# Create the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(
    vocab_size=vocab_size,   # 87 unique characters
    embed_size=128,          # each character becomes 128 numbers
    hidden_size=256,         # LSTM memory size
    num_layers=2             # 2 stacked LSTM layers
).to(device)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

LSTMModel(
  (embedding): Embedding(109, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=256, out_features=109, bias=True)
)

Total parameters: 963,565


In [5]:
import random

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

epochs = 20
print_every = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0
    count = 0

    # Shuffle batches each epoch
    random.shuffle(batches)

    for x, y in batches:
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)

        hidden = model.init_hidden(1, device)

        optimizer.zero_grad()
        output, hidden = model(x, hidden)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))
        loss.backward()

        # Gradient clipping - prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

        optimizer.step()
        total_loss += loss.item()
        count += 1

    # Perplexity = e^loss (measures how confused the model is)
    perplexity = torch.exp(torch.tensor(total_loss / count))

    if (epoch + 1) % print_every == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/count:.4f} | Perplexity: {perplexity:.2f}")

print("\nTraining complete!")

Epoch 2/20 | Loss: 1.5725 | Perplexity: 4.82
Epoch 4/20 | Loss: 1.4853 | Perplexity: 4.42
Epoch 6/20 | Loss: 1.4525 | Perplexity: 4.27
Epoch 8/20 | Loss: 1.4370 | Perplexity: 4.21
Epoch 10/20 | Loss: 1.4259 | Perplexity: 4.16
Epoch 12/20 | Loss: 1.4205 | Perplexity: 4.14
Epoch 14/20 | Loss: 1.4169 | Perplexity: 4.12
Epoch 16/20 | Loss: 1.4142 | Perplexity: 4.11
Epoch 18/20 | Loss: 1.4135 | Perplexity: 4.11
Epoch 20/20 | Loss: 1.4124 | Perplexity: 4.11

Training complete!


In [6]:
def generate_text(model, start_text, length=300, temperature=0.8):
    model.eval()
    chars = [char_to_idx[ch] for ch in start_text]
    input_tensor = torch.tensor(chars).unsqueeze(0).to(device)
    hidden = model.init_hidden(1, device)
    generated = start_text

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_tensor, hidden)
            output = output[:, -1, :] / temperature
            probs = torch.softmax(output, dim=-1)
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = idx_to_char[next_char_idx]
            generated += next_char
            input_tensor = torch.tensor([[next_char_idx]]).to(device)

    return generated

print(generate_text(model, "The mind ", length=200, temperature=0.5))
print()
print(generate_text(model, "Discipline ", length=200, temperature=0.8))
print()
print(generate_text(model, "A man who ", length=200, temperature=0.7))

The mind of the conversion and angry of the
fact more period, and the whole work of his considered the war that of the world
with the three of the state of the soul of the enemy’s present sound was not all


Discipline I was ranked to the coarse to made it
of the possible hand: "It is the vallen who can be too word the rest. It was that I couldn’t
be so tell your free and better that with the lost manner with the 

A man who is nothing at a
army was those who note, and there is continued in the discoveration. This human is the
high signs, and the whole including table to be a current way.

5. Where of the country is m


In [7]:
torch.save(model.state_dict(), 'lstm_quote_generator.pth')
print("Model saved!")

Model saved!
